para empezar darle a runtime, change runtime type y poner: T4 GPU

01. Configuración del entorno

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers accelerate

In [ ]:
!pip install -q bitsandbytes==0.48.2

In [ ]:
import importlib.metadata

print("bitsandbytes:", importlib.metadata.version("bitsandbytes"))

In [ ]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.16.1
Accelerate: 1.14.0


02. Autenticación Hugging Face

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '691518635cb559bdce2dd0fa', 'name': 'MBreadd', 'fullname': 'Saldaña Sánchez Luis Mario', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/196d6e7e55c0cf7cc9f1fc3c0aca02b3.svg', 'orgs': [{'type': 'org', 'id': '68ecc696bf1e3d158e71b7eb', 'name': 'MCP-1st-Birthday', 'fullname': 'MCP-1st-Birthday', 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/60d2dc1007da9c17c72708f8/N6eZhg0R7vFoxsPQODTOI.png'}], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-23T15:52:42.000Z'}}


03. Cargar tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "google/gemma-2-9b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer cargado correctamente")

Tokenizer cargado correctamente


04. Cargar Gemma 2 9B (8-bit)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()

print("Modelo cargado correctamente")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Modelo cargado correctamente


In [ ]:
print("GPU:", torch.cuda.get_device_name(0))

05. Descargar GSM8K

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print(dataset)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [ ]:
print("Train:", len(dataset["train"]))
print("Test:", len(dataset["test"]))

Train: 7473
Test: 1319


In [ ]:
print(dataset["test"][0])

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [ ]:
gsm8k_test = dataset["test"].select(range(20))

print("Problemas seleccionados:", len(gsm8k_test))

Problemas seleccionados: 20


In [ ]:
for i in range(3):
    print(f"\n--- Problema {i+1} ---")
    print(gsm8k_test[i]["question"])
    print("Respuesta esperada:", gsm8k_test[i]["answer"])


--- Problema 1 ---
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Respuesta esperada: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

--- Problema 2 ---
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Respuesta esperada: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3

--- Problema 3 ---
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
Respuesta esperada: The cost of the house and repairs came out to 80,000+50,000=$<<800

06. Cargar perturbaciones

07. Ejecutar experimento

8. Calcular entropía


09. Guardar resultados

contendido de prueba del modelo

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

prompt = "What is 2 + 2?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

respuesta = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(respuesta)

GPU: Tesla T4
What is 2 + 2?

[Answer 1]

2 + 


In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    output_scores=True,
    return_dict_in_generate=True
)

print("Número de pasos:", len(outputs.scores))
print("Shape del primer score:", outputs.scores[0].shape)
print("Tipo:", outputs.scores[0].dtype)

Número de pasos: 5
Shape del primer score: torch.Size([1, 256000])
Tipo: torch.float32


fin jeje